In [ ]:
%sql

--Cria catalog e schema
CREATE CATALOG IF NOT EXISTS lh_nautical;
CREATE SCHEMA IF NOT EXISTS lh_nautical.bronze;

In [ ]:
# Isolando a infraestrutura de conexão
from databricks.sdk.runtime import dbutils, spark 

db_user = dbutils.secrets.get(scope="sqlserver", key="username")
db_password = dbutils.secrets.get(scope="sqlserver", key="password")
db_host = dbutils.secrets.get(scope="sqlserver", key="host")
db_port = dbutils.secrets.get(scope="sqlserver", key="port")
db_database = dbutils.secrets.get(scope="sqlserver", key="db")

jdbc_url = f"jdbc:sqlserver://{db_host}:{db_port};databaseName={db_database}"

In [ ]:
import json
from pyspark.sql import functions as F

# 1. Carrega a lista de tabelas para ingestão integral na Bronze
with open('src/tabelas_config.json', 'r', encoding='utf-8') as f:
    lista_tabelas = json.load(f)

# 2. Carrega a política de mascaramento dinâmica externa
with open('src/governanca_config.json', 'r', encoding='utf-8') as f:
    politica_governanca = json.load(f)

print(f"Iniciando ingestão integral para as {len(lista_tabelas)} tabelas do ecossistema.")

# 3. Loop de processamento dinâmico
for tabela in lista_tabelas:
    print(f"\n[JDBC] Extraindo tabela: {tabela}")
    
    jdbc_options = {
        "url":                    jdbc_url,
        "dbtable":                tabela,
        "user":                   db_user,
        "password":               db_password,
        "driver":                 "com.microsoft.sqlserver.jdbc.SQLServerDriver",
        "encrypt":                "true",
        "trustServerCertificate": "true",
        "loginTimeout":           "30",
        "fetchsize":              "1000",
    }
    
    # Leitura da tabela original na íntegra
    df_temp = spark.read.format("jdbc").options(**jdbc_options).load()
    
    # Aplica o Hashing SHA-256 consultando o novo arquivo externo de governança
    if tabela in politica_governanca:
        for coluna_sensivel in politica_governanca[tabela]:
            if coluna_sensivel in df_temp.columns:
                print(f"  └─► [GOVERNANÇA] Aplicando SHA-256 na coluna: '{coluna_sensivel}'")
                df_temp = df_temp.withColumn(
                    coluna_sensivel, 
                    F.sha2(F.col(coluna_sensivel).cast("string"), 256)
                )
    
    # Persistência na camada Bronze gerenciada pelo Unity Catalog
    output_table = f"lh_nautical.bronze.{tabela}"
    df_temp.write.format("delta").mode("overwrite").saveAsTable(output_table)
    print(f"[SUCESSO] Tabela {output_table} carregada com segurança.")